# Week 6: CV Model — YOLOv8 on NEU-DET (Steel Surface Defects)

This notebook:
1. Installs dependencies
2. Downloads the dataset from Kaggle
3. Trains YOLOv8 in nano / small / medium / large variants
4. Compares results and picks the best model
5. Reports Precision, Recall, mAP@50, mAP@50-95
6. Shows training curves, confusion matrix, and sample predictions
7. Saves `best.pt`

**Runtime:** Go to `Runtime > Change runtime type > GPU` (T4 is fine) before running anything.

## 1. Install dependencies

In [2]:
!pip install -q ultralytics kaggle

import os
import yaml
import shutil
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

print('Setup complete.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Setup complete.


## 2. Upload and extract the dataset

Upload your `archive.zip` (the NEU-DET / neu-yolo dataset) here.

In [3]:
from google.colab import files

uploaded = files.upload()  # select archive.zip here
zip_name = list(uploaded.keys())[0]
print('Uploaded:', zip_name)

Saving archive.zip to archive.zip
Uploaded: archive.zip


In [4]:
os.makedirs('/content/data', exist_ok=True)
!unzip -q -o "{zip_name}" -d /content/data

# Inspect what we got
for p in Path('/content/data').rglob('*'):
    if p.is_dir():
        print(p)

/content/data/valid
/content/data/train
/content/data/valid/valid
/content/data/train/train
/content/data/valid/valid/images
/content/data/valid/valid/labels
/content/data/train/train/images
/content/data/train/train/labels


## 4. Build `data.yaml`

Ultralytics needs a YAML file pointing to the train/val image folders and listing class names. Your `archive.zip` has the confirmed structure `train/train/images`, `train/train/labels`, `valid/valid/images`, `valid/valid/labels`, with 6 defect classes (taken straight from the dataset's own `xml2yolo.py`): `crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches`.

In [5]:
data_root = Path('/content/data')

train_img_dir = data_root / 'train' / 'train' / 'images'
val_img_dir = data_root / 'valid' / 'valid' / 'images'

assert train_img_dir.exists(), f'Missing {train_img_dir} — check the printed folder tree above'
assert val_img_dir.exists(), f'Missing {val_img_dir} — check the printed folder tree above'

class_names = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']

data_yaml = {
    'train': str(train_img_dir),
    'val': str(val_img_dir),
    'nc': len(class_names),
    'names': class_names,
}

data_yaml_path = '/content/data/data.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)

print('Built data.yaml:')
print(Path(data_yaml_path).read_text())

Built data.yaml:
names:
- crazing
- inclusion
- patches
- pitted_surface
- rolled-in_scale
- scratches
nc: 6
train: /content/data/train/train/images
val: /content/data/valid/valid/images



## 5. Train YOLOv8 across model sizes

We train nano, small, medium, and large. On a free Colab T4, medium/large are considerably slower — start with `epochs=50` and raise it later if you have time budget. Each run logs to `runs/detect/<name>`.

In [1]:
MODEL_SIZES = ['n', 's', 'm', 'l']  # nano, small, medium, large
EPOCHS = 50
IMG_SIZE = 640

results_summary = {}

for size in MODEL_SIZES:
    print(f'\n=== Training YOLOv8{size} ===')
    model = YOLO(f'yolov8{size}.pt')
    run_name = f'yolov8{size}_neu'

    model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        name=run_name,
        patience=15,
        exist_ok=True,
    )

    metrics = model.val(data=data_yaml_path, name=f'{run_name}_val', exist_ok=True)

    results_summary[size] = {
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'run_dir': f'runs/detect/{run_name}',
        'weights': f'runs/detect/{run_name}/weights/best.pt',
    }

print('\nAll training runs complete.')


=== Training YOLOv8n ===


NameError: name 'YOLO' is not defined

## 6. Compare model sizes and pick the best

In [ ]:
import pandas as pd

df = pd.DataFrame(results_summary).T
df.index.name = 'model_size'
df = df[['precision', 'recall', 'map50', 'map50_95']]
print(df.to_string())

best_size = df['map50_95'].astype(float).idxmax()
print(f'\nBest model by mAP@50-95: YOLOv8{best_size}')
print(df.loc[best_size])

## 7. Save `best.pt`

In [ ]:
best_weights_src = results_summary[best_size]['weights']
final_best_path = '/content/best.pt'
shutil.copy(best_weights_src, final_best_path)
print(f'Saved best model (YOLOv8{best_size}) to {final_best_path}')

# Download to your machine
files.download(final_best_path)

## 8. Training curves

Ultralytics auto-saves a `results.png` per run — this is your training curve screenshot.

In [ ]:
best_run_dir = results_summary[best_size]['run_dir']

img = mpimg.imread(f'{best_run_dir}/results.png')
plt.figure(figsize=(14, 8))
plt.imshow(img)
plt.axis('off')
plt.title(f'Training Curves — YOLOv8{best_size}')
plt.show()

## 9. Confusion matrix

In [ ]:
img = mpimg.imread(f'{best_run_dir}/confusion_matrix.png')
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.title(f'Confusion Matrix — YOLOv8{best_size}')
plt.show()

## 10. Sample predictions

In [ ]:
best_model = YOLO(final_best_path)

val_dir = Path(yaml.safe_load(Path(data_yaml_path).read_text())['val'])
sample_images = list(val_dir.rglob('*.jpg'))[:6] or list(val_dir.rglob('*.png'))[:6]

pred_results = best_model.predict(source=sample_images, save=True, conf=0.25)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, res in zip(axes.flatten(), pred_results):
    ax.imshow(res.plot()[:, :, ::-1])  # BGR -> RGB
    ax.axis('off')
plt.suptitle('Sample Predictions')
plt.tight_layout()
plt.show()

## 11. Final metrics report

Compare against the assignment's thresholds:

| Metric | Good | Excellent |
|---|---|---|
| Precision | ≥ 0.80 | ≥ 0.90 |
| Recall | ≥ 0.60 | ≥ 0.75 |
| mAP@50 | ≥ 0.75 | ≥ 0.85 |
| mAP@50-95 | ≥ 0.40 | ≥ 0.50 |

In [ ]:
final = results_summary[best_size]
print(f"Best model: YOLOv8{best_size}")
print(f"Precision : {final['precision']:.3f}")
print(f"Recall    : {final['recall']:.3f}")
print(f"mAP@50    : {final['map50']:.3f}")
print(f"mAP@50-95 : {final['map50_95']:.3f}")